<h1>Multi Model RAG</h1>

<h3>Multi Model Rag is type of rag that can read multi document types like txt, pdf , png, docx,etc</h3>

In [4]:
import json
from typing import  List
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_chroma import Chroma 
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings

import os
import json
import base64
from google import genai
from google.genai import types

from dotenv import load_dotenv
import os

from google import genai

load_dotenv()

/home/aayush/miniconda3/envs/ragenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

<h2>function to load document </h2>

In [5]:
def partition_document(file_path:str):
    print("Extracting document ")
    elements=partition_pdf(
        filename=file_path,
        strategy='hi_res',
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True
    )
    print(f"exterated document content \n {len(elements)} elements")
    return elements

In [6]:
#define file_path and cal the partition_document function 
file_path='./doc/rag.pdf'
elements=partition_document(file_path)



Extracting document 


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 3956.54it/s]


exterated document content 
 389 elements


In [7]:
#set of type of elements that are extracted form document
set(str(type(el)) for el in elements)

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [8]:
#sample of element's attribute
elements[24].to_dict()

{'type': 'NarrativeText',
 'element_id': '7b01a905b1802baa8860cf1f900d2eb1',
 'text': 'In the fast-paced realm of digital transformation, businesses are increasingly pressured to innovate and boost efficiency to remain Abstract competitive and foster growth. Large Language Models (LLMs) have emerged as game-changers across industries, revolutionizing various sectors by harnessing extensive text data to analyze and generate human-like text. Despite their In the fast-paced realm of digital transformation, businesses are increasingly pressured to innovate and boost efficiency to remain impressive capabilities, LLMs often encounter challenges when dealing with domain-specific queries, potentially leading to competitive and foster growth. Large Language Models (LLMs) have emerged as game-changers across industries, inaccuracies in their outputs. In response, Retrieval-Augmented Generation (RAG) has emerged as a viable solution. By revolutionizing various sectors by harnessing extensive text

In [9]:
#select elements which are image
images=[el for el in elements if el.category=='Image']

In [10]:
images[0].to_dict()

{'type': 'Image',
 'element_id': '361e787cb4578e97f1c350764c17f5ba',
 'text': '- SEVIER',
 'metadata': {'coordinates': {'points': ((np.float64(188.77950243055554),
     np.float64(223.68675972222243)),
    (np.float64(188.77950243055554), np.float64(510.2726134722225)),
    (np.float64(432.33283041666664), np.float64(510.2726134722225)),
    (np.float64(432.33283041666664), np.float64(223.68675972222243))),
   'system': 'PixelSpace',
   'layout_width': 2646,
   'layout_height': 3611},
  'last_modified': '2026-09-10T11:06:43',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './doc',
  'filename': 'rag.pdf'}}

In [11]:
#select element which are tables
tables=[el for el in elements if el.category=='Table']
len(tables)

3

<h3>spliting document into chunk by title</h3>

In [12]:
def chunking(elements):
    print("chunking")

    chunk=chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2500,
        combine_text_under_n_chars=500
    )
    print("chunked creasted")
   
    return chunk

In [13]:
chunks=chunking(elements)
print(len(chunks))


chunking
chunked creasted
28


In [14]:
chunks[4].to_dict()

{'type': 'CompositeElement',
 'element_id': 'b1ee32db-9beb-4cd6-8cd6-84145b055881',
 'text': "analysts.\n\na — Retriever Dense/Sparse Input Text, Code, Image, Audio, Video Generator v Result Text, Code, Image, Audio, Video\n\nCould you provide me with ? re) any current construction- related business events? —|nput__ > LLM i 7 Analyst Output Without RAG Kk P As of my last update in January 2022, | don't have Output | = ct access to real-time information or current JF Domain data On March 2, 2023, an Austrian company announced (i.e., external a 400 million euro investment in constructing a database) tractor factory in Arada town, Ghioroc, Romania. e.g., News articles for business events\n\n(a) (b)\n\nFig. 1. (a) A generic RAG architecture, where users’ queries, potentially in different modalities (e.g., text, code, image, etc.), are inputted into both the retriever and the generator. The retriever scans for relevant data sources in storage, while the generator engages with the retrieval 

<h3>seperation  of table , text, images form chunk  </h3>
for the better processing to make sumarize , searchable text 

In [15]:
def seperate_type(chunk):
    content_data={
        'text':chunk.text,
        'table':[],
        'image':[],
        'types':['text']
    }

    if hasattr(chunk,'metadata') and hasattr(chunk.metadata,'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type=type(element).__name__

            if element_type=='Table':
                content_data['types'].append('table')
                table_html=getattr(element.metadata,'text_as_html',element.text)
                content_data['table'].append(table_html)

            elif element_type=="Image":
                if hasattr(element,'metadata') and hasattr(element.metadata,'image_base64'):
                    content_data['types'].append('image')
                    content_data['image'].append(element.metadata.image_base64)


    content_data['types']=list(set(content_data['types']))
    return content_data


In [16]:
cotanat=seperate_type(chunks[4])

In [17]:
print(cotanat)

{'text': "analysts.\n\na — Retriever Dense/Sparse Input Text, Code, Image, Audio, Video Generator v Result Text, Code, Image, Audio, Video\n\nCould you provide me with ? re) any current construction- related business events? —|nput__ > LLM i 7 Analyst Output Without RAG Kk P As of my last update in January 2022, | don't have Output | = ct access to real-time information or current JF Domain data On March 2, 2023, an Austrian company announced (i.e., external a 400 million euro investment in constructing a database) tractor factory in Arada town, Ghioroc, Romania. e.g., News articles for business events\n\n(a) (b)\n\nFig. 1. (a) A generic RAG architecture, where users’ queries, potentially in different modalities (e.g., text, code, image, etc.), are inputted into both the retriever and the generator. The retriever scans for relevant data sources in storage, while the generator engages with the retrieval outcomes, ultimately generating results across various modalities [6]; Fig. 1. (b) i

<h3>creating summary if table , image are present in chunk for making searchable </h3>

In [18]:
def create_ai_summary(text:str,tables:list[str],images:list[str]):



    prompt_text="""you are creating a searchable description for document content retrivial.

    CONTENT TO ANALYZE
    TEXT CONTENT:
    {text}

"""
    if tables:
        prompt_text+="TABLES:\n"
        for i, table in enumerate(tables):
            prompt_text+=f"table {i+1} :\n{table}\n\n"

            prompt_text+="""
Your Task:
Generate a comprehensive, serachable description that covers:
1.key facts,numbers, and data points from text and tables
2.Main topic and concepts discussed
3.Question this content could answer
4.Visual content analysis (charts,diagrams, pattern in images)
5. alternative search terms users might use

Make it detailed and searchable - prioritize findablility over
SEARCHABLE DISCRIPTION:

"""
    message_content=[{"type":"text","text":prompt_text}]

    for image_base64 in images:
        message_content.append({
            "type":"image_url",
            "image_url":{"url":f"data:image/jpeg;base64,{image_base64}"}
        })


    message=HumanMessage(content=message_content)

    api_key = os.getenv("GEMINI_API_KEY")
    
    if not api_key:
        raise ValueError("GEMINI_API_KEY environment variable is not set.")
    
    llm = genai.Client(api_key=api_key)
    
    response = llm.models.generate_content(
            model="gemini-3.6-flash",
            contents=message,

        )
    return response.text

In [19]:
def sumarize_chunks(chunks):
    print("sumarring chunks")

    langchains_documents=[]
    total_chunks=len(chunks)

    for i , chunk in enumerate(chunks):
        current_chunk=i+1
        print(f"processing {current_chunk}/{total_chunks}")

        content_data=seperate_type(chunk)

        print(f"      types found {content_data['types']}")
        print(f"       tables: {len(content_data['table'])} and images: {len(content_data['image'])}") 

        if content_data['image'] or content_data['table']:
            print("createing ai summarize ")

            try:
                enhanced_content=create_ai_summary(
                    content_data['text'],
                    content_data['table'],
                    content_data['image']
                )

                print("ai summary created successfully")
                print("..........."*9)

            except  Exception as e:
                print("ai summary faild")
                enhanced_content=content_data['text']
        else:
            print("using raw data")
            enhanced_content=content_data['text']

        doc=Document(
            page_content=enhanced_content,
            metadata={
                "orginal_content":json.dumps({
                    "raw_text":content_data['text'],
                    "table_html":content_data['table'],
                    'image_base64':content_data['image']
                })
            }
        )
        langchains_documents.append(doc)

    print("procseed langechange document")
    return langchains_documents
    

In [20]:
processed_chunks=sumarize_chunks(chunks)

sumarring chunks
processing 1/28
      types found ['image', 'text']
       tables: 0 and images: 3
createing ai summarize 


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ai summary faild
processing 2/28
      types found ['text']
       tables: 0 and images: 0
using raw data
processing 3/28
      types found ['text']
       tables: 0 and images: 0
using raw data
processing 4/28
      types found ['text']
       tables: 0 and images: 0
using raw data
processing 5/28
      types found ['image', 'text']
       tables: 0 and images: 2
createing ai summarize 
ai summary faild
processing 6/28
      types found ['text']
       tables: 0 and images: 0
using raw data
processing 7/28
      types found ['image', 'text']
       tables: 0 and images: 1
createing ai summarize 
ai summary faild
processing 8/28
      types found ['image', 'text']
       tables: 0 and images: 1
createing ai summarize 
ai summary faild
processing 9/28
      types found ['text']
       tables: 0 and images: 0
using raw data
processing 10/28
      types found ['text']
       tables: 0 and images: 0
using raw data
processing 11/28
      types found ['text', 'table']
       tables: 1 and im

In [21]:
len(processed_chunks)

28

<h3>creating vector database </h3>

using chroma database, we store embedded cunks and retrive using similarity consine function

In [26]:
def  create_vector_db(documents,presist_directory='db/chroma_db'):
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2")

    vectorstore=Chroma.from_documents(
        documents=documents,
        persist_directory=presist_directory,
        embedding=embedding_model,
        collection_metadata={'hnsw:space':'cosine'}
    )
    print("document stored oin vectorstore")
    return vectorstore

In [27]:
database=create_vector_db(processed_chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5199.65it/s]


document stored oin vectorstore


<h3> Retriveal function</h3>

you can change query according to your needs

In [28]:
query="How many publications related to RAG applications were recorded in the year 2023?"

def retrive_doc(query):
    retriveal=database.as_retriever(search_kwargs={"k":5})
    retrived_content=retriveal.invoke(query)
    print(retrived_content)
    return retrived_content

In [29]:
related_doc=retrive_doc(query)

[Document(id='26968775-18a0-407b-82b4-61734a0e2a3f', metadata={'orginal_content': '{"raw_text": "Fig. 2. Research Method\\n\\n3783\\n\\n3\\n\\n3784\\n\\nMuhammad Arslan et al. / Procedia Computer Science 246 (2024) 3781\\u20133790\\n\\n4\\n\\nArslan et al. / Procedia Computer Science 00 (2024) 000\\u2013000\\n\\nNumber of Publications Over the Years Number of Publications 2020 2022 2023 2024 Year\\n\\nFig. 3. Evolution of Research Publications on RAG Applications", "table_html": [], "image_base64": ["..."]}'}, page_content='Fig. 2. Research Method\n\n3783\n\n3\n\n3784\n\nMuhammad Arslan et al. / Procedia Computer Science 246 (2024) 3781–3790\n\n4\n\nArslan et al. / Procedia Computer Science 00 (2024) 000–000\n\nNumber of Publications Over the Years Number of Publications 2020 2022 2023 2024 Year\n\nFig. 3. Evolution of Research Publications on RAG Applications'), Document(id='bc61467d-adfc-4745-b437-5f1c8396a595', metadata={'orginal_content': '{"raw_text": "2. Background\\n\\nThe resea

In [30]:
related_doc

[Document(id='26968775-18a0-407b-82b4-61734a0e2a3f', metadata={'orginal_content': '{"raw_text": "Fig. 2. Research Method\\n\\n3783\\n\\n3\\n\\n3784\\n\\nMuhammad Arslan et al. / Procedia Computer Science 246 (2024) 3781\\u20133790\\n\\n4\\n\\nArslan et al. / Procedia Computer Science 00 (2024) 000\\u2013000\\n\\nNumber of Publications Over the Years Number of Publications 2020 2022 2023 2024 Year\\n\\nFig. 3. Evolution of Research Publications on RAG Applications", "table_html": [], "image_base64": ["..."]}'}, page_content='Fig. 2. Research Method\n\n3783\n\n3\n\n3784\n\nMuhammad Arslan et al. / Procedia Computer Science 246 (2024) 3781–3790\n\n4\n\nArslan et al. / Procedia Computer Science 00 (2024) 000–000\n\nNumber of Publications Over the Years Number of Publications 2020 2022 2023 2024 Year\n\nFig. 3. Evolution of Research Publications on RAG Applications'),
 Document(id='bc61467d-adfc-4745-b437-5f1c8396a595', metadata={'orginal_content': '{"raw_text": "2. Background\\n\\nThe rese

<h3>Generate answer using retrived_doc and LLM</h3>

In [31]:


def answer(chunks, query):
    try:
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("GEMINI_API_KEY environment variable is not set.")
            
        client = genai.Client(api_key=api_key)

        prompt = f"""Based on the following document please answer this question: {query}

CONTENT TO ANALYZE
"""
        for i, chunk in enumerate(chunks):
            prompt += f'____Document {i+1}____\n'

            if "orginal_content" in chunk.metadata:
                original_content = json.loads(chunk.metadata['orginal_content'])

                raw_text = original_content.get('raw_text', "")
                if raw_text:
                    prompt += f"TEXT:\n{raw_text}\n\n"

                table_html = original_content.get('tables', [])
                if table_html:
                    prompt += "TABLES:\n"
                    for j, table in enumerate(table_html):
                        prompt += f"Table {j+1}:\n{table}\n"

            prompt += "\n"
        
        prompt += """Please provide an answer using the text, tables, and images above. If the document doesn't contain sufficient information to answer the question, say "I don't have enough information to answer the question based on the provided document".
ANSWER:"""

        # Build contents list for the native Google GenAI SDK
        contents = [prompt]

        for chunk in chunks:
            if "orginal_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata['orginal_content'])
                images_base64 = original_data.get('images_base64', [])

                for image_base64 in images_base64:
                    image_bytes = base64.b64decode(image_base64)
                    contents.append(
                        types.Part.from_bytes(
                            data=image_bytes,
                            mime_type="image/jpeg"
                        )
                    )

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=contents,
        )

        return response.text

    except Exception as e:
        print(f"Answer generation failed: {e}")
        return None

In [32]:
result=answer(related_doc,query)
result

'Based on Document 5, **28 publications** related to RAG applications were recorded in the year 2023.'